In [2]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import pickle

In [3]:
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
data=data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

In [5]:
## Encode the categorical features
label_encoder_gender=LabelEncoder()
data["Gender"]=label_encoder_gender.fit_transform(data["Gender"])

In [7]:
## One hot encode the "Geography" column
onehot_encoder_geo=OneHotEncoder(handle_unknown="ignore")
geo_encoded=onehot_encoder_geo.fit_transform(data[["Geography"]]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(["Geography"]))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [8]:
##combine the one hot encoded columns with the original dataframe   
data=pd.concat([data.drop("Geography", axis=1), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [9]:
X=data.drop("EstimatedSalary", axis=1)
y=data['EstimatedSalary']

In [11]:
## Split the data into training and testing sets
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=42)



In [12]:
##Scale the features
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [13]:
# save the pickle files 
with open ('label_encoder_gender.pkl', 'wb') as f:
    pickle.dump(label_encoder_gender, f)
with open ('onehot_encoder_geo.pkl', 'wb') as f:
    pickle.dump(onehot_encoder_geo, f)
with open ('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)



In [15]:
## ANN regression model
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [16]:
# Biuld the model
model=Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    Dense(32, activation="relu"),
    Dense(1) ## Default activation for regression is linear
])

In [18]:
## compile the model
model.compile(optimizer="adam", loss="mean_absolute_error",metrics=["mae"])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [19]:
## logs
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

# setup tensorboard
log_dir="regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback=TensorBoard(log_dir=log_dir, histogram_freq=1)

In [20]:
# Setup early stopping
early_stopping_callback=EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

In [21]:
# Train the model
history=model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=100, callbacks=[tensorboard_callback, early_stopping_callback])

Epoch 1/100


250/250 [==============================] - 1s 3ms/step - loss: 100369.2891 - mae: 100369.2891 - val_loss: 98486.2109 - val_mae: 98486.2109
Epoch 2/100
250/250 [==============================] - 0s 2ms/step - loss: 99532.3438 - mae: 99532.3438 - val_loss: 96807.3750 - val_mae: 96807.3750
Epoch 3/100
250/250 [==============================] - 0s 2ms/step - loss: 96613.3594 - mae: 96613.3594 - val_loss: 92493.8828 - val_mae: 92493.8828
Epoch 4/100
250/250 [==============================] - 0s 2ms/step - loss: 90801.0078 - mae: 90801.0078 - val_loss: 85295.8906 - val_mae: 85295.8906
Epoch 5/100
250/250 [==============================] - 0s 2ms/step - loss: 82424.0234 - mae: 82424.0234 - val_loss: 76276.2812 - val_mae: 76276.2812
Epoch 6/100
250/250 [==============================] - 0s 2ms/step - loss: 72895.1562 - mae: 72895.1562 - val_loss: 67251.7812 - val_mae: 67251.7812
Epoch 7/100
250/250 [==============================] - 0s 1ms/step - loss: 64062.2461 - mae: 64062.246

In [23]:
%load_ext tensorboard
%tensorboard --logdir regressionlogs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6007 (pid 30056), started 0:00:32 ago. (Use '!kill 30056' to kill it.)

In [24]:
## Evaluate the model
test_loss, test_mae=model.evaluate(X_test, y_test)
print(f"Test MAE: {test_mae}")

63/63 [==============================] - 0s 1ms/step - loss: 50293.5625 - mae: 50293.5625
Test MAE: 50293.5625


In [25]:
model.save("regression_model.h5")

d:\GenAi Course\ANN Classification project\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
